# MuScriptor — Audio to Multi-Instrument Sheet Music
Official model: https://github.com/muscriptor/muscriptor

T4 GPU Colab notebook. Outputs multi-instrument MIDI, MusicXML, and (with MuseScore 4+) sheet-music PDFs. The published model weights are **CC BY-NC 4.0**.

Before running, accept the license on the selected Hugging Face model page: [small](https://huggingface.co/MuScriptor/muscriptor-small) / [medium](https://huggingface.co/MuScriptor/muscriptor-medium) / [large](https://huggingface.co/MuScriptor/muscriptor-large).

## 1. Install MuScriptor

In [ ]:
%pip -q install uv huggingface_hub
import subprocess
subprocess.run(["uv", "python", "install", "3.12"], check=True)
subprocess.run(["uv", "venv", "/content/muscriptor_env", "--python", "3.12"], check=True)
subprocess.run(["uv", "pip", "install", "--python", "/content/muscriptor_env/bin/python", "git+https://github.com/muscriptor/muscriptor.git"], check=True)
muscriptor_cli = "/content/muscriptor_env/bin/muscriptor"

## 2. Hugging Face login
Use a Hugging Face read token with access to the accepted model license.

In [ ]:
from huggingface_hub import login
login()

## 3. Upload original song

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess

uploads = files.upload()
input_dir = Path("/content/muscriptor_input")
input_dir.mkdir(parents=True, exist_ok=True)
audio_paths = []
for filename, data in uploads.items():
    source = input_dir / filename
    source.write_bytes(data)
    audio_paths.append(source)
    print(source)

## 4. Install MuseScore 4+ for sheet PDF export
MuScriptor requires MuseScore 4 or newer for `--format sheets`. This cell fetches a Linux x86-64 AppImage from the official MuseScore GitHub releases and extracts it for headless Colab execution.

In [ ]:
import json
import os
import subprocess
import urllib.request
from pathlib import Path

releases_api = "https://api.github.com/repos/musescore/MuseScore/releases/latest"
req = urllib.request.Request(releases_api, headers={"User-Agent": "colab-muscriptor"})
with urllib.request.urlopen(req) as response:
    release = json.load(response)
assets = release["assets"]
matches = [a for a in assets if a["name"].endswith(".AppImage") and "x86_64" in a["name"]]
if not matches:
    raise RuntimeError("No x86_64 MuseScore AppImage found in the official latest release")
asset = matches[0]
appimage_path = Path("/content/MuseScore4.AppImage")
urllib.request.urlretrieve(asset["browser_download_url"], appimage_path)
appimage_path.chmod(0o755)
subprocess.run([str(appimage_path), "--appimage-extract"], check=True, cwd="/content", stdout=subprocess.DEVNULL)
musescore_executable = Path("/content/squashfs-root/AppRun")
os.environ["MUSCRIPTOR_MUSESCORE"] = str(musescore_executable)
print("MuseScore:", musescore_executable)

## 5. Transcribe original audio into a full score
Model `medium` is MuScriptor's official default; select `small` if desired. This invokes the documented sheet export CLI.

In [ ]:
#@title MuScriptor 채보 — 실시간 로그
MODEL = "medium" #@param ["small", "medium", "large"]

from pathlib import Path
import os
import subprocess

score_root = Path("/content/muscriptor_scores")
score_root.mkdir(parents=True, exist_ok=True)

env = dict(os.environ)
env["QT_QPA_PLATFORM"] = "offscreen"
env["MUSCRIPTOR_MUSESCORE"] = str(musescore_executable)
env["PYTHONUNBUFFERED"] = "1"

for audio in audio_paths:
    destination = score_root / audio.stem
    command = [
        muscriptor_cli, "transcribe", str(audio),
        "--model", MODEL,
        "--format", "sheets",
        "--output", str(destination)
    ]
    print(f"Processing: {audio.name}", flush=True)
    with subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"MuScriptor failed (exit code {returncode})")
    print(f"Completed: {destination}", flush=True)

## 6. List generated scores and download
Includes whole-score PDF, per-instrument PDFs, MusicXML, and MIDI.

In [ ]:
from pathlib import Path
from google.colab import files

all_scores = sorted(p for p in score_root.rglob("*") if p.is_file())
for i, path in enumerate(all_scores):
    print(f"{i:02d}  {path.relative_to(score_root)}")

DOWNLOAD_INDEX = 0 #@param {type:"integer"}
files.download(str(all_scores[DOWNLOAD_INDEX]))